## Réseau cristallin du $\text{NaNO}_2$

- BODART Quentin - 70362100 
- DEJAIE Lisa - 75692300
- STIEVENARD Alexandre - 79822300  
- VAN EETVELDE Eric - 38962300  

#### 1. Imports, téléchargement de la structure et fonctions utiles

In [ ]:
### IMPORTS ###
try:
    import numpy as np
    from numpy.linalg import norm
    import py3Dmol
    from mp_api.client import MPRester
    from pymatgen.symmetry.analyzer import SpacegroupAnalyzer,Structure
    from pymatgen.analysis.diffraction.xrd import XRDCalculator
    import pandas as pd
    import matplotlib.pyplot as plt
    from pymatgen.electronic_structure.plotter import plot_brillouin_zone
    from pymatgen.symmetry.bandstructure import HighSymmKpath



except ImportError as e:
    print(f"module or submodule \"{e.name}\" not found.\
        \nPlease run the following command in the same folder as this notebook to ensure all needed modules are installed:\
        \npip install -r requirements.txt")
    raise SystemExit

In [ ]:
### TELECHARGEMENT
mp_key = "cLLBrCi8gU7SqhnPKS4NJqSKB4Y6d0gD"
mp_id = "mp-2964"

with MPRester(mp_key) as mpr: structure = mpr.get_structure_by_material_id(mp_id)
sga = SpacegroupAnalyzer(structure)

In [ ]:
### FONCTIONS ###
def print_title(title:str):
    print('\n' + title + '\n' + "-"*len(title))

def get_angle(v1:np.ndarray, v2:np.ndarray) -> float:
    """Retourne l'angle en degrés entre deux vecteurs 3D"""
    return np.rad2deg(np.arccos(np.dot(v1,v2) / (norm(v1)*norm(v2))))

def py3Dmol_visualise(structure:Structure):
    """Crée une fenêtre de visualisation interactive de la structure donnée"""
    view = py3Dmol.view(
        width=700, 
        height=500,
        data=structure.to(fmt="cif"),
        format="cif",
        style={"sphere": {"scale": 0.3}, "stick": {"radius": 0.1}}
    )
    view.addUnitCell()
    view.zoomTo()
    view.show()

#### 2. Vecteurs de bases, type de maille, système cristallin et  groupe ponctuel

##### 2.1 Visualisation 

2.1.1. Maille primitive

In [ ]:
py3Dmol_visualise(structure)

2.1.2. Maille conventionelle

In [ ]:
py3Dmol_visualise(structure.to_conventional())

##### 2.2 Vecteurs de bases 

In [ ]:
### FONCTIONS HELPER ###
def lattice_vectors_df(matrix: np.ndarray) -> pd.DataFrame:
    """
    matrix : tableau de taille (3,3) où les lignes correspondent aux vecteurs a, b, c en coordonnées cartésiennes.
    Retourne un DataFrame contenant les composantes et les normes.
    """
    labels = ["a", "b", "c"]
    rows = []
    for i, lab in enumerate(labels):
        vec = matrix[i]
        rows.append({
            "vecteur": lab,
            "x": float(vec[0]),
            "y": float(vec[1]),
            "z": float(vec[2]),
            "norme": float(norm(vec)),
        })
    df = pd.DataFrame(rows, columns=["vecteur", "x", "y", "z", "norme"])
    return df

def lattice_angles_df(matrix: np.ndarray) -> pd.DataFrame:
    """
    Calcule alpha = angle(b,c), beta = angle(c,a), gamma = angle(a,b) en degrés
    en utilisant la fonction existante get_angle(v1, v2).
    """
    alpha = get_angle(matrix[1], matrix[2])
    beta  = get_angle(matrix[2], matrix[0])
    gamma = get_angle(matrix[0], matrix[1])
    return pd.DataFrame({
        "angle": ["α", "β", "γ"],
        "valeur (°)": [alpha, beta, gamma]
    })

def show_lattice(title: str, matrix: np.ndarray, precision: int = 2) -> None:
    """
    Affiche un titre puis les DataFrames des vecteurs du réseau (avec leurs normes)
    et des angles associés, avec un arrondi à la précision donnée.
    """
    print_title(title)
    df_vec = lattice_vectors_df(matrix).round(precision)
    df_ang = lattice_angles_df(matrix).round(precision)

    display(df_vec.round(precision))
    display(df_ang.round(precision))

def verify_direct_reciprocal_compatible(direct, reciprocal, tol=1e-9):
    """
    Vérifie que les vecteurs du réseau direct: d1,d2,d3 sont compatibles avec les 
    vecteurs du réseau réciproque: r1,r2,r3 en vérifiant la relation:
        d_i · r_j = 2π * δ_ij

    Paramètres
    ----------
    d1,d2,d3 : array-like (3,)
        Vecteurs du réseau direct.
    r1,r2,r3 : array-like (3,)
        Vecteurs du réseau réciproque.
    tol : float
        Tolérance sur les produits scalaires.
    ----------
    """
    # Construire les matrices (3x3) : chaque ligne est un vecteur
    direct = np.array(direct, dtype=float)
    reciprocal = np.array(reciprocal, dtype=float)

    # Vérifications de forme
    if direct.shape != (3, 3) or reciprocal.shape != (3, 3):
        raise ValueError(
            f"Chaque vecteur doit être de dimension 3. "
            f"Shapes obtenues: direct={direct.shape}, reciprocal={reciprocal.shape}"
        )

    # Matrice des produits scalaires: (i,j) = d_i · r_j
    dot_matrix = direct @ reciprocal.T

    target = 2.0 * np.pi * np.eye(3)
    error_matrix = dot_matrix - target
    max_abs_error = float(np.max(np.abs(error_matrix)))

    ok = bool(max_abs_error <= tol)
    if (ok):
        print("Les vecteurs du réseau direct et réciproque sont compatibles")
        return
    print("Erreur: Les vecteurs du réseau direct et réciproque sont compatibles")
    return

def K_from_hkl(h, k, l, recp_matrix):
    "Récupère le vecteur K du réseau réciproque à partir des indices de miller et de la matrice contenant les vecteurs du réseau réciproque"
    return h * recp_matrix[0] + k * recp_matrix[1] + l * recp_matrix[2]


In [ ]:
### RESEAU DIRECT PRIMITIF ###
matrix = np.array(structure.lattice.matrix)
show_lattice("Vecteurs de base du réseau direct de la maille primitive (en Å)", matrix)

### RESEAU RECIPROQUE PRIMITIF ###

recp_matrix = np.array(structure.lattice.reciprocal_lattice.matrix)
show_lattice("Vecteurs de base du réseau réciproque de la maille primitive (en Å⁻¹)", recp_matrix)

verify_direct_reciprocal_compatible(matrix, recp_matrix)

In [ ]:
### RESEAU DIRECT CONVENTIONNEL ###
matrix = np.array(structure.to_conventional().lattice.matrix)
show_lattice("Vecteurs de base du réseau direct de la maille conventionelle (en Å)", matrix)

### RESEAU RECIPROQUE CONVENTIONNEL ###

recp_matrix = np.array(structure.to_conventional().lattice.reciprocal_lattice.matrix)
show_lattice("Vecteurs de base du réseau réciproque de la maille conventionnelle (en Å⁻¹)", recp_matrix)

verify_direct_reciprocal_compatible(matrix, recp_matrix)

In [ ]:
group_symbol_to_french = {
    "P" : "Primitive",
    "I" : "Centrée",
    "A" : "Bases centrées (A)",
    "B" : "Bases centrées (B)",
    "C" : "Bases centrées (C)",
    "F" : "Faces centrées"
}
crystal_system_to_french = {
    "cubic" : "Cubique",
    "hexagonal" : "Hexagonal",
    "monoclinic" : "Monoclinique",
    "orthorhombic" : "Orthorombique",
    "tetragonal" : "Tétragonal",
    "triclinic" : "Triclinique",
    "trigonal" : "Trigonal"
}

### TYPE DE MAILLE ###
print_title("Type de maille")
print(group_symbol_to_french[sga.get_space_group_symbol()[0]])

### SYSTEME CRYSTALLIN ###
print_title("Système cristallin")
print(crystal_system_to_french[sga.get_crystal_system()])

### GROUPE PONCTUEL ###
print_title("Groupe ponctuel")
print(sga.get_point_group_symbol())

#### Opérations de symmétrie

In [ ]:
#### HELPER FUNCTIONS ####
def draw_lattice_axes(ax, origin=(0.05,0.05), length=0.15):
    
    x0, y0 = origin
    
    # a axis
    ax.annotate(
        "", 
        xy=(x0+length, y0), 
        xytext=(x0, y0),
        arrowprops=dict(arrowstyle="->", linewidth=2)
    )
    ax.text(x0+length+0.02, y0, "a", va='center')
    
    # b axis
    ax.annotate(
        "",
        xy=(x0, y0+length),
        xytext=(x0, y0),
        arrowprops=dict(arrowstyle="->", linewidth=2)
    )
    ax.text(x0, y0+length+0.02, "b", ha='center')
    
    # c axis (out of plane)
    ax.text(x0-0.01, y0-0.03, "⊙ c")

Le groupe ponctel: mm2

Dans un systeme orthorombique, en suivant la notation de Hermann-Mauguin, les trois caractères décrivent la symétrie le long des trois axes $a,b$ et $c$.

Nous avons donc:
- Une réflexion de normale $a$
- Une réflexion de normale $b$
- Un axe de rotation d'ordre 2 le long de $c$



In [ ]:
conventional = sga.get_conventional_standard_structure() 

# Récupération de la position des atomes dans la maille conventionelle
atoms_conv = []

for site in conventional:
    x, y, z = site.frac_coords 
    atoms_conv.append({
        "element": site.specie.symbol,
        "pos": (x, y, z)
    })

print_title("Atomes dans la cellule conventionelle")
for i, atom in enumerate(atoms_conv):
    x, y, z = atom["pos"]
    print(f"{i:2d} | {atom['element']:2s} | ({x:.8f}, {y:.8f}, {z:.8f})")

##### Réflexion de normale a:
$$ (x,y,z) \rightarrow (\bar{x},y,z) $$

Nous montrons l'effet de cette opération sur le **Na** (#0)  situé en:
\begin{equation}
    \begin{bmatrix*}
        0.50 \\
        0.50 \\
        0.087
    \end{bmatrix*}
    \text{(Lettre de Wyckoff: e, multiplicité: 8)}
\end{equation}

##### Réflexion de normale b:
$$ (x,y,z) \rightarrow (x,\bar{y},z) $$

Nous montrons l'effet de cette opération sur le **O** (#4)  situé en:
\begin{equation*}
    \begin{bmatrix}
        0.00 \\
        0.20 \\
        1.00
    \end{bmatrix}
    \text{(Lettre de Wyckoff: d, multiplicité: 4)}
\end{equation*}

##### Rotation $C_2$ autour de c:
$$ (x,y,z) \rightarrow (\bar{x},\bar{y},z) $$

Nous montrons l'effet de cette opération sur le **Na** (#0)  situé en:
\begin{equation*}
    \begin{bmatrix}
        0.50 \\
        0.50 \\
        0.087
    \end{bmatrix}
    \text{(Lettre de Wyckoff: e, multiplicité: 8)}
\end{equation*}

L'effet de ces trois opérations sont reprises dans un même plot en projetant sur un plan de normale $\vec{c}$

In [ ]:
# --- Selection des atoms ---
symmetry_atoms = [atoms_conv[0], atoms_conv[4], atoms_conv[7]]

original_positions = [np.array(atom["pos"][:2], dtype=float) for atom in symmetry_atoms]

# --- Position des atoms symmétriques ---
copied_positions = []

# 1) Reflection 1 : (x,y) -> (-x,y)
p1 = original_positions[0].copy()
p1[0] = -p1[0]
copied_positions.append(p1)

# 2) Reflection 2 : (x,y) -> (x,-y)
p2 = original_positions[1].copy()
p2[1] = -p2[1]
copied_positions.append(p2)

# 3) Rotation C2 : (x,y) -> (-x,-y)
p3 = original_positions[2].copy()
p3[0] = -p3[0]
p3[1] = -p3[1]
copied_positions.append(p3)

# --- Plot ---
fig, ax = plt.subplots(figsize=(7, 7))

ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_aspect("equal")

ax.set_xticks(np.linspace(-1, 1, 11))
ax.set_yticks(np.linspace(-1, 1, 11))
ax.grid(True, linestyle="--", alpha=0.4)

# Plans Mirroirs
ax.axvline(0, linestyle="--", color="red", linewidth=1.5, label="mirroir $\perp$ a")
ax.axhline(0, linestyle="--", color="green", linewidth=1.5, label="mirroir $\perp$ b")

# Centre de rotation
ax.scatter(0, 0, marker="x", s=120, color="purple")
ax.text(0.03, 0.03, r"$C_2$", color="purple")

# Couleurs
orig_colors = ["blue", "blue", "blue"]
copy_colors = ["red", "green", "purple"]
op_labels = [r"$m_a$", r"$m_b$", r"$C_2$"]
atom_names = ["Na", "O#4", "O#7"]

for i in range(3):
    original_pos = original_positions[i]
    copy_pos = copied_positions[i]

    # Atome original
    ax.scatter(original_pos[0], original_pos[1], s=500, color=orig_colors[i], zorder=3)
    ax.text(original_pos[0], original_pos[1], atom_names[i], ha="center", va="center", color="white")

    # Atomes symmétrique
    ax.scatter(copy_pos[0], copy_pos[1], s=500, color=copy_colors[i], zorder=3)
    ax.text(copy_pos[0], copy_pos[1], atom_names[i], ha="center", va="center", color="white")

    # Label de l'opération
    ax.text(copy_pos[0] + 0.05, copy_pos[1] + 0.05, op_labels[i], color=copy_colors[i], fontsize=12)

draw_lattice_axes(ax, origin=(-0.82, -0.82), length=0.18)

ax.set_xlabel("a")
ax.set_ylabel("b")
ax.set_title("Opérations du groupe mm2 dans le plan ab")
ax.legend()

plt.show()

#### 3. Visualisation de la zone de Brillouin

In [ ]:
primitive = structure.get_primitive_structure()

print_title("Première zone de Brillouin")

fig = plot_brillouin_zone(structure.lattice.reciprocal_lattice)


#### 4. Diffractograme

<img src="figures/reflection_conditions.png" width="150">  

Source: *International tables for crystallography : Volume A*

In [ ]:
def gaussian_broaden(two_theta_sticks, intensities, tt_grid, sigma=0.20):
    """
    Construit une courbe continue à partir des pics discrets de diffraction.
    """
    sigma = sigma / (0.9 * np.sqrt(2.0 * np.log(2.0))) 
    y = np.zeros_like(tt_grid, dtype=float)

    for tt, I in zip(two_theta_sticks, intensities):
        y += I * np.exp(-0.5 * ((tt_grid - tt) / sigma) ** 2)

    return y

In [ ]:
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.analysis.diffraction.xrd import XRDCalculator

# On va imposer le groupe à la structure pour réduire les erreurs numériques
sga = SpacegroupAnalyzer(structure, symprec=1e-2, angle_tolerance=5)

# Calcul du pattern de diffraction
wavelength = 1.54060
calc = XRDCalculator(wavelength=wavelength)    
pattern = calc.get_pattern(conventional, scaled=True,two_theta_range=(0,50))

two_theta=np.array(pattern.x)
intensity=np.array(pattern.y)
hkls=np.array(pattern.hkls)

# Masque pour retirer les pics d'intensité trop faible (<2,5%)
treshold = 0.025 * intensity.max()  
mask = intensity > treshold
# Filtrage des pics
two_theta_filtered = two_theta[mask]
intensity_filtered = intensity[mask]
hkls_filtered = hkls[mask]

# Tracer la courbe gausienne continue
tt_grid = np.linspace(0, 50, 6000)  
y_cont = gaussian_broaden(two_theta_filtered, intensity_filtered, tt_grid, sigma=0.20)

plt.figure()
plt.plot(tt_grid, y_cont)                
plt.vlines(two_theta, 0, intensity, linewidth=1)  
plt.xlabel(r"$2\theta$ (deg)")
plt.ylabel("Intensité (normalisée)")
plt.title("XRD pattern")
plt.show()





Le diffractogramme présente les 3 premiers pics aux angles approximatifs suivants: 31, 33, 46 degrés.
\
Grossisons le spectrogramme pour voir plus précisément quels plans (hkl) génèrent ces pics.

In [ ]:
print_title("Spectrogramme grossi de 30 à 47 degrés")
calc.show_plot(conventional, two_theta_range=(30,47))

In [ ]:
peaks_data = [
    {
        "Pic": 1,
        "2θ (°)": f"{two_theta_filtered[0]:.2f} / {two_theta_filtered[1]:.2f}",
        "Intensité": f"{intensity_filtered[0]:.2f} / {intensity_filtered[1]:.2f}",
        "hkl": f"{hkls_filtered[0][0]['hkl']} (m={hkls_filtered[0][0]['multiplicity']}) / {hkls_filtered[1][0]['hkl']} (m={hkls_filtered[1][0]['multiplicity']})"
    },
    {
        "Pic": 2,
        "2θ (°)": f"{two_theta_filtered[2]:.2f}",
        "Intensité": f"{intensity_filtered[2]:.2f}",
        "hkl": f"{hkls_filtered[2][0]['hkl']} (m={hkls_filtered[2][0]['multiplicity']})"
    },
    {
        "Pic": 3,
        "2θ (°)": f"{two_theta_filtered[3]:.2f} / {two_theta_filtered[4]:.2f}",
        "Intensité": f"{intensity_filtered[3]:.2f} / {intensity_filtered[4]:.2f}",
        "hkl": f"{hkls_filtered[3][0]['hkl']} (m={hkls_filtered[3][0]['multiplicity']}) / {hkls_filtered[4][0]['hkl']} (m={hkls_filtered[4][0]['multiplicity']})"
    }
]

df_peaks = pd.DataFrame(peaks_data)
display(df_peaks)
print("m: multiplicité")
print("*l'intensité est normalisée avec le pic le plus intense prenant la valeur 100")

##### Analyse et interprétation des résultats

Le premier pic est generé par les plans (1,0,1) et (1,1,0). La contribution au pic des deux plans est caractérisée par un angle légèrement différent du fait que $d_{101} < d_{110}$.  
En effet:

In [ ]:
def d_hkl(G):
    return 2*np.pi / np.linalg.norm(G)

# Vecteurs du réseau réciproque
K_101 = K_from_hkl(1,0,1,recp_matrix)
K_110 = K_from_hkl(1,1,0,recp_matrix)

d_101 = d_hkl(K_101)
d_110 = d_hkl(K_110)

print(f"d_101: {d_101:.3f}")
print(f"d_110: {d_110:.3f}")

# Calculons la différence d'angle a partir de Bragg
theta_101 = np.rad2deg(np.arcsin(wavelength/(2*d_101)))
theta_110 = np.rad2deg(np.arcsin(wavelength/(2*d_110)))
print(f"Delta 2*theta = {2*(theta_110 - theta_101):.3f}")

print("Nous retrouvons bien la différence d'angle du diffractograme")


En ce qui concerne la différence importante d'intensité entre les plans (101) et (110) elle peut être expliquée par l'analyse du facteur de structure de la maille. Avant de calculer celui-ci, nous avons besoins des facteurs de forme atomique.
Nous pouvons les estimer sous l'hypothèse d'une densité électronique sphérique symmétrique autour du noyau en utilisant la formule suivante:
$$
f(|\vec{K}|) = \sum_{i=1}^{4} a_i \exp\left(-b_i \left(\frac{K}{4\pi}\right)^2 \right) + c
$$

avec les valeurs des coefficients:

| Element | a1 | b1 | a2 | b2 | a3 | b3 | a4 | b4 | c |
|--------|----|----|----|----|----|----|----|----|----|
| Na | 4.7626 | 3.285 | 3.1736 | 8.8422 | 1.2674 | 0.3136 | 1.1128 | 129.424 | 0.676 |
| O  | 3.0485 | 13.2771 | 2.2868 | 5.7011 | 1.5463 | 0.3239 | 0.867 | 32.9089 | 0.2508 |
| N  | 12.2126 | 0.0057 | 3.1322 | 9.8933 | 2.0125 | 28.9975 | 1.1663 | 0.5826 | -11.529 |

La formule et les coefficients sont issus de:
[Graz Center of Physics](https://lampz.tugraz.at/~hadley/ss1/crystaldiffraction/atomicformfactors/formfactors.php)


In [ ]:
atoms = ["Na", "O", "N"]

a = {
    "Na": [4.7626, 3.1736, 1.2674, 1.1128],
    "O":  [3.0485, 2.2868, 1.5463, 0.867],
    "N":  [12.2126, 3.1322, 2.0125, 1.1663]
}

b = {
    "Na": [3.285, 8.8422, 0.3136, 129.424],
    "O":  [13.2771, 5.7011, 0.3239, 32.9089],
    "N":  [0.0057, 9.8933, 28.9975, 0.5826]
}

c = {
    "Na": 0.676,
    "O": 0.2508,
    "N": -11.529
}

def atomic_form_factor(Knorm, atom):
    """
    Calcule le facteur d'onde atomique sur base de la norme d'un vecteur K 
    du réseau réciproque et de l'atome considéré
    """
    s = Knorm / (4*np.pi)
    
    ai = a[atom]
    bi = b[atom]
    ci = c[atom]

    f = 0
    for i in range(4):
        f += ai[i] * np.exp(-bi[i] * s**2)

    return f + ci

À partir de l'estimation du facteur de forme atomique $f_j(|\mathbf{K}|)$, on peut calculer le facteur de structure de la maille :

$$
S_{\mathbf{G}} = \sum_{j=1}^{N} f_j(|\mathbf{G}|)\, e^{i\,\mathbf{G}\cdot \mathbf{r}_j}
$$

In [ ]:
def structure_factor(h,k,l, Knorm, atoms):
    """
    Calcule le facteur de structure pour un vecteur K
    atoms: list de dictionnaires comme: {"element": "Na", "pos"= (x,y,z)}
    """

    S = 0.0 + 0.0j

    for atom in atoms:
        element = atom["element"]
        x,y,z = atom["pos"]

        fj = atomic_form_factor(Knorm, element)
        phase = np.exp(1j * 2 * np.pi * (h*x + k*y + l*z))

        S += fj*phase
        
    return S

print_title("Résultats")
 
S_101 = structure_factor(1,0,1,np.linalg.norm(K_101),atoms_conv)
S_110 = structure_factor(1,1,0,np.linalg.norm(K_110),atoms_conv)

# Multiplicité des pics
m_101 = hkls_filtered[0][0]['multiplicity']
m_110 = hkls_filtered[1][0]['multiplicity']

I_101 = m_101*abs(S_101)**2
I_110 = m_110*abs(S_110)**2

print("S_101 =", S_101)
print("S_110 =", S_110)
print("I_101 =", I_101)
print("I_110 =", I_110)
print(f"Intensité relative de I_101 par rapport à I_110 = {(I_101 / I_110)*100:.1f}")

L'intensité relative correspond à quelques dizièmes à celle du diffractogramme. La même analyse peut être effectuée pour les autres pics. Nous pouvons par exemple vérifier l'extinction du pic (100) qui ne répond pas à la condition : $$ h = 2n$$

In [ ]:
K_100 = K_from_hkl(1,0,0,recp_matrix)
S_100 = structure_factor(1,0,0,np.linalg.norm(K_100), atoms_conv)
I_100 = abs(S_100)**2
print(f"Intensité relative de I_100 par rapport à I_110 = {(I_100 / I_110)*100}")